# 00 — Click Capture

We are no longer drawing maps.

We are asking maps questions.

But before we can ask anything, we need to capture input. This notebook is about one thing only:

```text
click → print → store → visualize
```

No geometry. No algorithms. Just: *user clicks, we receive coordinates*. Get that working first. Everything else builds on it.

## Setup

In [ ]:
from pathlib import Path
from ipyleaflet import Map, basemaps
from ipywidgets import Output

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Ready. Data directory:", DATA_DIR)

## How Click Events Work

ipyleaflet maps emit events when the user interacts with them. The `on_interaction` method registers a callback function that runs every time an interaction occurs.

The callback receives a dictionary of keyword arguments (`**kwargs`) describing what happened:

```python
# What kwargs looks like on a click:
{
    'type': 'click',
    'coordinates': [lat, lon],   # ← the click location
    ...other fields...
}
```

The callback fires on **every** interaction — mousemove, zoom, click, etc. — so always check `kwargs['type'] == 'click'` before doing anything.

> **Note**: `coordinates` returns `[lat, lon]` — latitude first, longitude second. This is ipyleaflet's convention and it is **different** from GeoJSON. We'll address this head-on in notebook 01.

## Step 1 — Print on Click

In [ ]:
out = Output()

m = Map(center=(33, 44), zoom=4, basemap=basemaps.CartoDB.Positron)

def handle_click(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs['coordinates']   # [lat, lon]
        with out:
            out.clear_output(wait=True)
            print(f"Click received!")
            print(f"  Raw coordinates: {coords}")
            print(f"  lat={coords[0]:.4f}  lon={coords[1]:.4f}")

m.on_interaction(handle_click)

display(m, out)

Click anywhere on the map above. The output updates below the map.

Notice:
- `kwargs['type']` filters out mousemove and other noise
- `kwargs['coordinates']` gives `[lat, lon]` — latitude is index 0
- Each click overwrites the previous output (`clear_output`)

## Step 2 — Store the Last Click

In [ ]:
# Store last click in a mutable container so the callback can write to it
# (using a list avoids the 'nonlocal' keyword for simplicity)
last_click = [None]

out2 = Output()
m2 = Map(center=(33, 44), zoom=4, basemap=basemaps.CartoDB.Positron)

def store_click(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs['coordinates']   # [lat, lon]
        last_click[0] = coords
        with out2:
            out2.clear_output(wait=True)
            print(f"Stored: lat={coords[0]:.4f}, lon={coords[1]:.4f}")

m2.on_interaction(store_click)
display(m2, out2)

In [ ]:
# Read the stored value at any time after clicking
if last_click[0] is not None:
    print(f"Last click: {last_click[0]}")
else:
    print("No click recorded yet — click the map above first.")

## Step 3 — Accumulate Multiple Clicks

For most analyses we need more than one click — a series of points to query against a dataset. Append each click to a list.

In [ ]:
click_log = []   # grows with each click

out3 = Output()
m3 = Map(center=(33, 44), zoom=4, basemap=basemaps.CartoDB.Positron)

def log_click(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs['coordinates']   # [lat, lon]
        click_log.append(coords)
        with out3:
            out3.clear_output(wait=True)
            print(f"{len(click_log)} click(s) recorded:")
            for i, c in enumerate(click_log):
                print(f"  {i+1}.  lat={c[0]:.4f}  lon={c[1]:.4f}")

m3.on_interaction(log_click)
display(m3, out3)

## Step 4 — The Full Event Structure

When debugging click events, print the entire `kwargs` dict to see everything available. Run this once to understand the full payload.

In [ ]:
import json

out4 = Output()
m4 = Map(center=(33, 44), zoom=4, basemap=basemaps.CartoDB.Positron)

def inspect_event(**kwargs):
    if kwargs.get('type') == 'click':
        with out4:
            out4.clear_output(wait=True)
            print("Full click event kwargs:")
            for key, val in kwargs.items():
                print(f"  {key!r:20}  →  {val!r}")

m4.on_interaction(inspect_event)
display(m4, out4)

## Exercise A

Create a map that **prints the click coordinates** in a clean format:

```
Click #1:  lat=35.6950  lon=51.3880
Click #2:  lat=24.6880  lon=46.6750
Click #3:  lat=40.4170  lon=-3.7030
```

1. Each click appends a new line (do **not** clear previous output).
2. Include a click counter (`#1`, `#2`, ...) in each line.
3. Click at least 3 different locations to verify it works.

In [ ]:
from ipyleaflet import Map, basemaps
from ipywidgets import Output

# Counter and log
# Map with on_interaction handler
# Print formatted output, appending (not clearing) each click
# Your code here

## Exercise B

Store the **last clicked location** and print it on demand.

1. Create a map with a click handler that stores the most recent click.
2. After clicking the map, run a separate cell that reads and prints the stored value.
3. Add a guard: if no click has been recorded yet, print `"No click yet"` instead of crashing.

In [ ]:
from ipyleaflet import Map, basemaps
from ipywidgets import Output

# Map + click handler (stores last click)
# Your code here

In [ ]:
# Run this cell after clicking the map above
# Print stored click or "No click yet"
# Your code here

---

## Check Your Understanding

**1.** What data does a click event provide, and what format does it come in?

**2.** Why do we need to *store* the click result rather than just printing it?

```python
# No code needed — answer in your own words
```

## Next

In [01 — Point Representation](./01_Point_Representation.ipynb), we confront the first gotcha head-on: ipyleaflet gives `[lat, lon]` but GeoJSON requires `[lon, lat]` — and we build a clean conversion so the rest of the module always uses the right order.